In [ ]:
import os, gc, time, warnings, re
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.stats import spearmanr
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import Ridge

warnings.filterwarnings('ignore')
t_start = time.time()

FEATURE_FRACTION   = 0.6
SEEDS              = [42, 123, 456, 789, 2024]
RIDGE_ALPHA        = 5000
W_LGB              = 0.50
W_RIDGE            = 0.50
TARGET_STD         = 0.001500
INTER_STD          = 0.000948
CLIP_Z             = 5.0
EPS                = 1e-6
N_CHUNKS           = 20
ICIR_GOLD          = 3.0
ICIR_ENG           = 2.0
TOPK_GOLD_FALLBACK = 10

DATA_DIR = '/kaggle/input/short-horizon-return-prediction-challenge-by-i-rage'
OUT_DIR  = '/kaggle/working'

In [ ]:
print('Loading data...', flush=True)
train = pd.read_parquet(f'{DATA_DIR}/train.parquet')
test  = pd.read_parquet(f'{DATA_DIR}/test.parquet')

feat_cols = [c for c in train.columns if c not in {'ID', 'TARGET', 'CV_GROUP'}]
all_feat  = [c for c in feat_cols if c != 'SO3_T' and c in test.columns]

train_day      = train['SO3_T'].round(5).astype(str).values
test_day       = test['SO3_T'].round(5).astype(str).values
train_days_set = set(np.unique(train_day))
te_ids         = test['ID'].values
n_test         = len(test)

y_raw      = train['TARGET'].values.astype(np.float64)
lo_y, hi_y = np.percentile(y_raw, 1), np.percentile(y_raw, 99)
y_wins     = np.clip(y_raw, lo_y, hi_y)
y_dm       = y_wins.copy()
for d in np.unique(train_day):
    m = train_day == d
    y_dm[m] = y_wins[m] - y_wins[m].mean()

groups5        = pd.qcut(pd.Series(train['SO3_T'].values), q=5,
                          labels=False, duplicates='drop').values.astype(np.int32)
train_sort_idx = np.argsort(train['ID'].values)

tr_raw = train[all_feat].fillna(0).values.astype(np.float32)
te_raw = test[all_feat].fillna(0).values.astype(np.float32)
del test; gc.collect()
print(f'Train: {tr_raw.shape}, Test: {te_raw.shape}', flush=True)

In [ ]:
print('ICIR feature selection (scipy spearmanr per chunk)...', flush=True)
train_s = train.sort_values('ID').reset_index(drop=True)
cs = len(train_s) // N_CHUNKS
spearman_icir = {}
for col in all_feat:
    cics = []
    for i in range(N_CHUNKS):
        ch    = train_s.iloc[i*cs:(i+1)*cs]
        v     = ch[col].fillna(ch[col].median()).values
        t     = ch['TARGET'].values
        valid = ~np.isnan(v)
        if valid.sum() < 200: continue
        ic, _ = spearmanr(v[valid], t[valid])
        if not np.isnan(ic): cics.append(ic)
    if len(cics) >= 5:
        mic = float(np.mean(cics)); sic = float(np.std(cics)) + 1e-8
        spearman_icir[col] = dict(mean_ic=mic, icir=mic/sic, abs_icir=abs(mic/sic),
                                  ic_pos=float(np.mean([v > 0 for v in cics])))

orig_gold_feats = sorted([k for k, v in spearman_icir.items()
                           if v['abs_icir'] >= ICIR_GOLD and v['ic_pos'] in (0.0, 1.0)],
                          key=lambda x: -spearman_icir[x]['abs_icir'])
print(f'orig_gold={len(orig_gold_feats)}', flush=True)

del train, train_s; gc.collect()
print('Freed train DataFrame.', flush=True)

In [ ]:
def parse_feature(fname):
    m = re.match(r'^(.+?)(_LagT(\d+))$', fname)
    if m: return m.group(1), f'LagT{m.group(3)}'
    return fname, 'base'

base_to_lags = {}
for f in all_feat:
    base, lag = parse_feature(f)
    base_to_lags.setdefault(base, {})[lag] = f
features_with_lags = {b: lags for b, lags in base_to_lags.items()
                      if 'base' in lags and 'LagT1' in lags}
base_max_icir  = {b: max(spearman_icir.get(f, {'abs_icir': 0})['abs_icir']
                         for f in lags.values())
                  for b, lags in features_with_lags.items()}
important_bases = {b for b, v in base_max_icir.items() if v >= ICIR_ENG}

gold_lag1 = [f for f in all_feat if f.endswith('_LagT1')]
lag1_icir = {c: spearman_icir[c]['abs_icir'] for c in gold_lag1 if c in spearman_icir}
top_lag1  = sorted(lag1_icir.keys(), key=lambda x: -lag1_icir[x])[:10]
print(f'important_bases={len(important_bases)}, top_lag1={len(top_lag1)}', flush=True)

In [ ]:
print('Computing per-day z-score stats...', flush=True)
global_mean = tr_raw.mean(0).astype(np.float64)
global_std  = tr_raw.std(0).astype(np.float64)
global_std[global_std < 1e-8] = 1.0

day_stats = {}
for d in np.unique(train_day):
    m = train_day == d
    if m.sum() < 50: continue
    x  = tr_raw[m].astype(np.float64)
    mu = x.mean(0); sg = x.std(0); sg[sg < 1e-8] = 1.0
    day_stats[d] = (mu, sg)
print(f'day_stats: {len(day_stats)} days with >=50 samples', flush=True)

def apply_pd_norm(raw, days):
    out = np.empty_like(raw, dtype=np.float32)
    for d in np.unique(days):
        m = days == d
        mu, sg = day_stats.get(d, (global_mean, global_std))
        z = (raw[m].astype(np.float64) - mu) / sg
        out[m] = np.clip(z, -CLIP_Z, CLIP_Z).astype(np.float32)
    return out

print('Applying pd norm to train...', flush=True)
X_tr_pd = apply_pd_norm(tr_raw, train_day)
print('Applying pd norm to test...', flush=True)
X_te_pd = apply_pd_norm(te_raw, test_day)
print(f'X_tr_pd: {X_tr_pd.shape}, X_te_pd: {X_te_pd.shape}', flush=True)

In [ ]:
def fast_spearman_icir(X_matrix, y_target, feat_names, n_chunks=N_CHUNKS):
    chunk_size = len(y_target) // n_chunks
    X_sorted   = X_matrix[train_sort_idx]
    y_sorted   = y_target[train_sort_idx]
    all_ics    = np.zeros((n_chunks, X_sorted.shape[1]), dtype=np.float64)
    for i in range(n_chunks):
        s = i*chunk_size; e = (i+1)*chunk_size
        X_ch   = X_sorted[s:e].astype(np.float64)
        y_ch   = y_sorted[s:e].astype(np.float64)
        X_rank = np.argsort(np.argsort(X_ch, axis=0), axis=0).astype(np.float64)
        y_rank = np.argsort(np.argsort(y_ch)).astype(np.float64)
        X_m    = X_rank - X_rank.mean(0); y_m = y_rank - y_rank.mean()
        X_sd   = np.sqrt((X_m**2).sum(0)); y_sd = np.sqrt((y_m**2).sum())
        X_sd[X_sd < 1e-10] = 1e-10
        if y_sd > 1e-10:
            all_ics[i] = (X_m.T @ y_m) / (X_sd * y_sd)
    out = {}
    for ci, name in enumerate(feat_names):
        ics   = all_ics[:, ci]; valid = ~np.isnan(ics)
        if valid.sum() < 5: continue
        mic   = float(np.mean(ics[valid])); sic = float(np.std(ics[valid])) + 1e-8
        out[name] = dict(mean_ic=mic, icir=mic/sic, abs_icir=abs(mic/sic),
                         ic_pos=float(np.mean(ics[valid] > 0)))
    return out

print('Engineering lag features...', flush=True)
eng_names = []; eng_tr = []; eng_te = []

def add_feat(name, tr_v, te_v):
    eng_names.append(name)
    eng_tr.append(tr_v.ravel().astype(np.float32))
    eng_te.append(te_v.ravel().astype(np.float32))

for base_name, lags in features_with_lags.items():
    if base_name not in important_bases: continue
    idx_b  = all_feat.index(lags['base'])
    idx_l1 = all_feat.index(lags['LagT1'])
    has_l2 = 'LagT2' in lags; has_l3 = 'LagT3' in lags
    b_tr   = X_tr_pd[:, idx_b].astype(np.float64);  b_te  = X_te_pd[:, idx_b].astype(np.float64)
    l1_tr  = X_tr_pd[:, idx_l1].astype(np.float64); l1_te = X_te_pd[:, idx_l1].astype(np.float64)
    add_feat(f'past_T1_{base_name}',   b_tr - l1_tr,   b_te - l1_te)
    den_tr = np.abs(b_tr) + EPS; den_te = np.abs(b_te) + EPS
    add_feat(f'relchg_T1_{base_name}', np.clip(l1_tr/den_tr,-10,10), np.clip(l1_te/den_te,-10,10))
    add_feat(f'lvlxchg_T1_{base_name}', b_tr*l1_tr, b_te*l1_te)
    add_feat(f'abschg_T1_{base_name}', np.abs(l1_tr), np.abs(l1_te))
    add_feat(f'signchg_T1_{base_name}', np.sign(l1_tr), np.sign(l1_te))
    if has_l2:
        idx_l2 = all_feat.index(lags['LagT2'])
        l2_tr  = X_tr_pd[:, idx_l2].astype(np.float64); l2_te = X_te_pd[:, idx_l2].astype(np.float64)
        add_feat(f'accel_T1T2_{base_name}',   l2_tr - l1_tr,   l2_te - l1_te)
        add_feat(f'consist_T1T2_{base_name}', np.sign(l1_tr)*np.sign(l2_tr), np.sign(l1_te)*np.sign(l2_te))
        add_feat(f'past_T2_{base_name}',   b_tr - l2_tr,   b_te - l2_te)
    if has_l3:
        idx_l3 = all_feat.index(lags['LagT3'])
        l3_tr  = X_tr_pd[:, idx_l3].astype(np.float64); l3_te = X_te_pd[:, idx_l3].astype(np.float64)
        add_feat(f'longshort_{base_name}', l3_tr - l1_tr, l3_te - l1_te)
        if has_l2:
            add_feat(f'accel2_{base_name}', l3_tr - 2*l2_tr + l1_tr, l3_te - 2*l2_te + l1_te)
            net_tr = np.abs(b_tr - l3_tr); net_te = np.abs(b_te - l3_te)
            vol_tr = np.abs(b_tr-l1_tr)+np.abs(l1_tr-l2_tr)+np.abs(l2_tr-l3_tr)+EPS
            vol_te = np.abs(b_te-l1_te)+np.abs(l1_te-l2_te)+np.abs(l2_te-l3_te)+EPS
            kauf_tr = np.clip(net_tr/vol_tr, 0, 1); kauf_te = np.clip(net_te/vol_te, 0, 1)
            add_feat(f'kaufer_{base_name}',  kauf_tr, kauf_te)
            add_feat(f'skaufer_{base_name}', np.sign(b_tr-l3_tr)*kauf_tr, np.sign(b_te-l3_te)*kauf_te)

for i in range(len(top_lag1)):
    for j in range(i+1, len(top_lag1)):
        fi = all_feat.index(top_lag1[i]); fj = all_feat.index(top_lag1[j])
        add_feat(f'xchg_{top_lag1[i]}_x_{top_lag1[j]}',
                 X_tr_pd[:,fi].astype(np.float64) * X_tr_pd[:,fj].astype(np.float64),
                 X_te_pd[:,fi].astype(np.float64) * X_te_pd[:,fj].astype(np.float64))

for base_name, lags in features_with_lags.items():
    if base_name not in important_bases: continue
    idx_b = all_feat.index(lags['base'])
    b_tr  = X_tr_pd[:,idx_b].astype(np.float64); b_te = X_te_pd[:,idx_b].astype(np.float64)
    add_feat(f'log_{base_name}', np.sign(b_tr)*np.log1p(np.abs(b_tr)), np.sign(b_te)*np.log1p(np.abs(b_te)))
    add_feat(f'sq_{base_name}',  b_tr**2, b_te**2)

print(f'Engineered {len(eng_names)} features. Stacking...', flush=True)
if eng_tr:
    X_tr_eng = np.column_stack(eng_tr).astype(np.float32)
    X_te_eng = np.column_stack(eng_te).astype(np.float32)
else:
    X_tr_eng = np.zeros((tr_raw.shape[0], 0), dtype=np.float32)
    X_te_eng = np.zeros((te_raw.shape[0], 0), dtype=np.float32)
del eng_tr, eng_te; gc.collect()
np.nan_to_num(X_tr_eng, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
np.nan_to_num(X_te_eng, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
print(f'X_tr_eng: {X_tr_eng.shape}', flush=True)

In [ ]:
print('ICIR on engineered features...', flush=True)
if X_tr_eng.shape[1] > 0:
    icir_eng = fast_spearman_icir(X_tr_eng, y_raw, eng_names)
else:
    icir_eng = {}

icir_all = {**spearman_icir, **icir_eng}
gold_all = {k: v for k, v in icir_all.items()
            if v['abs_icir'] >= ICIR_GOLD and v['ic_pos'] in (0.0, 1.0)}

gold_orig_feats2 = sorted([k for k in gold_all if k in all_feat],
                           key=lambda x: -gold_all[x]['abs_icir'])
gold_eng_feats   = sorted([k for k in gold_all if k in eng_names],
                           key=lambda x: -gold_all[x]['abs_icir'])
print(f'gold_orig={len(gold_orig_feats2)} gold_eng={len(gold_eng_feats)}', flush=True)

orig_gold_idx = [all_feat.index(f) for f in gold_orig_feats2]
eng_gold_idx  = [eng_names.index(f) for f in gold_eng_feats]

X_tr_orig_gold = X_tr_pd[:, orig_gold_idx]
X_te_orig_gold = X_te_pd[:, orig_gold_idx]

if eng_gold_idx:
    X_tr_gold = np.hstack([X_tr_orig_gold, X_tr_eng[:, eng_gold_idx]])
    X_te_gold = np.hstack([X_te_orig_gold, X_te_eng[:, eng_gold_idx]])
else:
    X_tr_gold = X_tr_orig_gold
    X_te_gold = X_te_orig_gold

del X_tr_pd, X_te_pd, X_tr_eng, X_te_eng, X_tr_orig_gold, X_te_orig_gold; gc.collect()
print(f'Final gold matrix: {X_tr_gold.shape}', flush=True)

In [ ]:
print('Training LGB (feature_fraction=0.6, 5 seeds x 5 folds)...', flush=True)
BASE_PARAMS = dict(
    objective='regression', metric='rmse',
    num_leaves=63, learning_rate=0.05,
    bagging_fraction=0.8, bagging_freq=1,
    min_child_samples=50, lambda_l1=0.1, lambda_l2=1.0,
    feature_fraction=FEATURE_FRACTION,
    n_jobs=-1, verbose=-1,
)

gkf   = GroupKFold(n_splits=len(np.unique(groups5)))
folds = list(gkf.split(X_tr_gold, y_dm, groups=groups5))

lgb_te = np.zeros(n_test, dtype=np.float64)
for s in SEEDS:
    params = dict(BASE_PARAMS)
    params['seed'] = s; params['feature_fraction_seed'] = s; params['bagging_seed'] = s
    seed_pred = np.zeros(n_test, dtype=np.float64)
    for fi, (tri, vai) in enumerate(folds):
        dt  = lgb.Dataset(X_tr_gold[tri], label=y_dm[tri], free_raw_data=True)
        dv  = lgb.Dataset(X_tr_gold[vai], label=y_dm[vai], reference=dt, free_raw_data=True)
        mdl = lgb.train(params, dt, num_boost_round=500, valid_sets=[dv],
                        callbacks=[lgb.early_stopping(500, verbose=False), lgb.log_evaluation(0)])
        seed_pred += mdl.predict(X_te_gold, num_iteration=mdl.best_iteration) / len(folds)
        print(f'  seed={s} fold={fi} best_iter={mdl.best_iteration}', flush=True)
        del dt, dv, mdl; gc.collect()
    lgb_te += seed_pred

lgb_te = (lgb_te / len(SEEDS)).astype(np.float32)
del X_tr_gold, X_te_gold; gc.collect()
print(f'LGB done. Elapsed: {(time.time()-t_start)/60:.1f} min', flush=True)

In [ ]:
print('Running per-day Ridge (pd normalization, alpha=5000)...', flush=True)
top10_gold    = orig_gold_feats[:TOPK_GOLD_FALLBACK]
ic_arr10      = np.array([spearman_icir[f]['mean_ic'] for f in top10_gold], dtype=np.float64)
top10_idx     = [all_feat.index(f) for f in top10_gold]
global_mean10 = tr_raw[:, top10_idx].mean(0).astype(np.float64)
global_std10  = tr_raw[:, top10_idx].std(0).astype(np.float64)
global_std10[global_std10 < 1e-8] = 1.0

ridge_te = np.zeros(n_test, dtype=np.float64)
n_ov = 0; n_nw = 0
for d in np.unique(test_day):
    te_mask    = test_day == d
    te_idx_arr = np.where(te_mask)[0]
    if d in train_days_set:
        tr_mask = train_day == d
        n_tr_d  = tr_mask.sum()
        if n_tr_d < 20:
            X_te_10 = te_raw[te_mask][:, top10_idx].astype(np.float64)
            sg_g    = np.where(global_std10 < 1e-8, 1.0, global_std10)
            X_te_z  = np.clip((X_te_10 - global_mean10) / sg_g, -CLIP_Z, CLIP_Z)
            ridge_te[te_idx_arr] = X_te_z @ ic_arr10
            n_nw += 1; continue
        X_tr_d = tr_raw[tr_mask].astype(np.float64)
        X_te_d = te_raw[te_mask].astype(np.float64)
        m_d    = X_tr_d.mean(0); s_d = X_tr_d.std(0); s_d[s_d < 1e-8] = 1.0
        X_tr_z = np.clip((X_tr_d - m_d) / s_d, -CLIP_Z, CLIP_Z)
        X_te_z = np.clip((X_te_d - m_d) / s_d, -CLIP_Z, CLIP_Z)
        y_tr_d = y_raw[tr_mask].astype(np.float64)
        y_tr_w = np.clip(y_tr_d, np.percentile(y_tr_d, 1), np.percentile(y_tr_d, 99))
        mdl    = Ridge(alpha=RIDGE_ALPHA, fit_intercept=True)
        mdl.fit(X_tr_z, y_tr_w)
        ridge_te[te_idx_arr] = mdl.predict(X_te_z)
        n_ov += 1
    else:
        X_te_10 = te_raw[te_mask][:, top10_idx].astype(np.float64)
        sg_g    = np.where(global_std10 < 1e-8, 1.0, global_std10)
        X_te_z  = np.clip((X_te_10 - global_mean10) / sg_g, -CLIP_Z, CLIP_Z)
        ridge_te[te_idx_arr] = X_te_z @ ic_arr10
        n_nw += 1

ridge_te = ridge_te.astype(np.float32)
print(f'Ridge done. overlap={n_ov} new_days={n_nw}. Elapsed: {(time.time()-t_start)/60:.1f} min', flush=True)

In [ ]:
def auto_scale(p, std):
    s = p.std()
    return p * (std / s) if s > 1e-10 else p

def finalize(pred, target_std=INTER_STD):
    p = pred.astype(np.float64).copy()
    p -= p.mean()
    return auto_scale(p, target_std)

lgb_fin   = finalize(lgb_te,   INTER_STD)
ridge_fin = finalize(ridge_te, INTER_STD)
blend     = W_LGB * lgb_fin + W_RIDGE * ridge_fin
scaled    = finalize(blend, TARGET_STD)

submission = pd.DataFrame({'ID': te_ids, 'TARGET': scaled.astype(np.float32)})
submission.to_csv(f'{OUT_DIR}/submission.csv', index=False)
print(f'Saved submission.csv  shape={submission.shape}  std={scaled.std():.6f}', flush=True)
print(f'Total runtime: {(time.time()-t_start)/60:.1f} min', flush=True)
submission.head()